In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



In [71]:
# loading the dataset
df = pd.read_csv("/content/train.txt",sep = ';', header = None, names = ['text', 'emotion'])

In [72]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [73]:
df.shape

(16000, 2)

In [74]:
df.isnull().sum()

,0
text,0
emotion,0


In [75]:
df.duplicated().sum()

np.int64(1)

In [76]:
df.drop_duplicates(inplace = True)

In [77]:
df.duplicated().sum()

np.int64(0)

In [78]:
df.shape

(15999, 2)

In [79]:
unique_emotion = df['emotion'].unique()

In [80]:
emotion_dict = {}
i = 0
for emo in unique_emotion:
    emotion_dict[emo] = i
    i += 1
df['emotion'] = df['emotion'].map(emotion_dict)

In [81]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [82]:
# lower the text
df['text'] = df['text'].apply(lambda x: x.lower())

In [83]:
# remove punctuation
import string
def remove_punctuation(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

df['text'] = df['text'].apply(remove_punctuation)


In [84]:
# remove the number (optional depends on the situation)
def remove_number(txt):
    new = ''
    for i in txt:
        if not i.isdigit():
            new += i
    return new

df['text'] = df['text'].apply(remove_number)


In [85]:
# remove emojis (all existing ones are represented by ascii)
def remove_emoji(txt):
    new = ''
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emoji)

In [86]:
# remove stopwords(stopwords in ml doesnot add any meaning to the sentence but useful in deep learning )
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [87]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [88]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [89]:
stopwords = (stopwords.words('english'))
len(stopwords)

198

In [90]:
def remove_stopwords(txt):
  words = word_tokenize(txt)
  cleaned = []
  for i in words:
    if not i in stopwords:
      cleaned.append(i)
  return ' '.join(cleaned)

df['text'] = df['text'].apply(remove_stopwords)

In [91]:
# after removing stopwords
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [92]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [93]:
X = df['text']
y = df['emotion']


In [94]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train

,text
676,refers course though cant help feeling somehow...
14180,would feel awkward someone tells feelings towards
7077,feel like probably would liked book little bit...
13004,ive slowed take time listen child moment feel ...
10264,feel vital rlsh gimmick needs
...,...
13418,love leave reader feeling confused slightly de...
5390,feel delicate
860,starting feel little stressed
15796,feel acceptable music criticisms reinforce con...


In [95]:
from sklearn.feature_extraction.text import CountVectorizer

Vectorizer = CountVectorizer()

X_train_bow = Vectorizer.fit_transform(X_train)

X_test_bow = Vectorizer.transform(X_test)

X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 115778 stored elements and shape (12799, 13354)>

In [100]:
from sklearn.feature_extraction.text import TfidfVectorizer
Vectorizer = TfidfVectorizer()
X_train_tfidf = Vectorizer.fit_transform(X_train)
X_test_tfidf = Vectorizer.transform(X_test)
X_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 115778 stored elements and shape (12799, 13354)>

In [101]:
from sklearn.linear_model import LogisticRegression
model_bow = LogisticRegression()
model_bow.fit(X_train_bow,y_train)


LogisticRegression()

In [102]:
model_tfidf = LogisticRegression()
model_tfidf.fit(X_train_tfidf, y_train)

LogisticRegression()

In [103]:
y_pred_bow = model_bow.predict(X_test_bow)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

In [104]:
from sklearn.metrics import accuracy_score
print("bow_accuracy: ",accuracy_score(y_test, y_pred_bow))
print("tfidf_accuracy: ",accuracy_score(y_test, y_pred_tfidf))

bow_accuracy:  0.89625
tfidf_accuracy:  0.86125
